In [ ]:
from src.metric_contract import METRIC_CONTRACTS
from src.utilitis import get_model_id
import numpy as np
import pandas as pd

In [ ]:
def aggregate_persisted_metrics(df):
    required = {'execucoes_iguais', *(metric.parquet_column for metric in METRIC_CONTRACTS if metric.parquet_column)}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Parquet executado incompatível; colunas ausentes: {sorted(missing)}")

    total = len(df)
    correct = int((df['execucoes_iguais'] == True).sum())
    rows = [{
        'key': 'execution_accuracy', 'code': 'EX',
        'value': correct / total if total else None,
        'available': total > 0, 'denominator': total, 'numerator': correct,
    }]
    for metric in METRIC_CONTRACTS[1:]:
        values = pd.to_numeric(df[metric.parquet_column], errors='raise').dropna()
        if not np.isfinite(values).all():
            raise ValueError(f"Métrica não finita: {metric.parquet_column}")
        denominator = len(values)
        rows.append({
            'key': metric.key, 'code': metric.code,
            'value': float(values.mean()) if denominator else None,
            'available': denominator > 0, 'denominator': denominator,
            'numerator': None,
        })
    return pd.DataFrame(rows).set_index('code')

In [ ]:
# Ajuste conscientemente modelos, bibliotecas e seed antes da análise.
models = ["Qwen2.5-Coder-7B-Instruct", "Qwen3-32B"]
# bibliotecas = ["vannaAi_exemplos", "vannaAi_contexto_exemplos"]
bibliotecas = ["rawModel_exemplos"]
seed = 42

In [ ]:
def load_metric_reports(db_name):
    reports = []
    for model_name in models:
        model_id = get_model_id(model_name)
        for biblioteca in bibliotecas:
            data_file = f"resources/out/{db_name}/{model_id}/queries_geradas_{biblioteca}_{seed}_executado.parquet"
            metrics = aggregate_persisted_metrics(pd.read_parquet(data_file)).reset_index()
            metrics.insert(0, 'biblioteca', biblioteca)
            metrics.insert(0, 'modelo', model_name)
            metrics.insert(0, 'database', db_name)
            reports.append(metrics)
    return pd.concat(reports, ignore_index=True) if reports else pd.DataFrame()

In [ ]:
# As 13 métricas são lidas do Parquet executado; somente EX continua derivada de execucoes_iguais.
load_metric_reports("sih_database")

In [ ]:
load_metric_reports("datasus")